# 7 · Jev Evals: grade 100% of your LLM's answers

## The problem

Your RAG bot answers thousands of questions a day. Some answers invent facts, some drift off-topic, some quietly refuse,
some answer half the question. You only find out when a customer complains.

The usual fix is **LLM-as-judge**: ask a big model to grade each answer. It works, but the judge costs about as much as
the model it grades, so teams grade a 1% sample and hope the other 99% look the same.

## Why Jev

Grading is a **judgment**, not writing. Each part of a rubric is a typed question with a typed answer: *is every claim
supported?* (yes/no), *how on-topic is it?* (a position on a scale). That is exactly what a **System One** model does:
typed, **calibrated** answers in about **100 ms** for a **fraction of a cent**, all rubric questions in **one** call.
Cheap enough to grade **every** response. And because Jev reports its own confidence, you can send the
*uncertain grades* to a human: an eval of the eval.

## What we'll build

```
(question, context, answer) ──► Jev: 4 rubric questions in ONE call ──► ordered rules in Python ──► verdict
                                 • faithful (Noul)                        1. non-answer?             PASS / REVISE / FAIL
                                 • relevant (Score 0..2)                  2. off-topic?              + every flag that tripped
                                 • complete (Score 0..2)                  3. hallucination?          + "review" if Jev was unsure
                                 • refuses  (Noul)                        4. incomplete?
```

## What you'll learn
- How to turn an eval rubric into typed Jev questions (`Noul` and `Score`) over a dict `state`.
- Why the **order** of your verdict rules is a design decision, and how to keep every failing signal visible anyway.
- How to use Jev's `confidence` as a **meta-eval**: flag grades a human should double-check.
- What grading every answer actually costs.

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads `TYPESAFE_API_KEY` from the `.env` file at the repo root (it searches this folder and its parents),
- creates the Jev client `jev`,
- imports the question types we'll use: `Noul` (a yes/no probability) and `Score` (a position on an ordered rubric),
- sets `PRICE_PER_M`, Jev's published price per million **input** tokens (output is free). We use it to show what grading costs.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, Score

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # Jev does the grading
PRICE_PER_M = 0.042      # dollars per 1M input tokens (output tokens are free)

print("Jev client ready")

Jev client ready


## Step 1 · The rubric, as typed questions

Every answer is graded on four dimensions. Each one is a single Jev question:

| Key | Type | What it returns | Read it as |
|---|---|---|---|
| `faithful` | `Noul` | `.noul`, P(yes), 0 to 1 | near 1 = every claim is backed by the context |
| `relevant` | `Score` | `.score` from 0 to 2, plus `.confidence` | 0 off-topic, 1 partial, 2 fully on-topic |
| `complete` | `Score` | `.score` from 0 to 2, plus `.confidence` | 0 misses most, 2 covers everything the context supports |
| `refuses` | `Noul` | `.noul`, P(yes) | near 1 = the answer declined instead of trying |

**Two Jev ideas appear here for the first time:**
- **`Noul`** answers a yes/no question with a *probability*: `.noul` is P(yes), from 0 to 1. You decide where to cut.
- **`Score`** takes an ordered rubric in `criteria`: the first line is level 0, the next level 1, and so on. Jev returns a
  *continuous* position on that scale (1.4 means "between partial and full, closer to partial") plus a `.confidence`.

**The state is a dict.** We will send `{"question": ..., "context": ..., "answer": ...}`. A name in backticks inside an
instruction, like `` `answer` ``, refers to that key of the dict. That is how Jev knows which text is which.

This cell only **defines** the rubric. Nothing is sent to Jev yet.

> **Workshop note:** the first draft asked *"Is `answer` a non-answer, refusal, or hedge rather than a real attempt?"*.
> Jev rated the off-topic answer *"Paris is a wonderful city..."* about **0.95** on that, because an off-topic reply is,
> in a sense, a non-answer. That blurred two different failures. The question now asks only about **declining**
> (*"decline, refuse, or say it cannot help"*), so off-topic answers are caught by `relevant` instead. Wording is the
> model's whole world: make each question mean one thing.

In [2]:
RUBRIC = {
    "faithful": Noul(
        instructions="Is EVERY factual claim in `answer` supported by `context`? Answer no if it adds any fact not in the context.",
    ),
    "relevant": Score(
        instructions="How well does `answer` address `question`?",
        criteria=["Off-topic", "Partially addresses it", "Directly and fully on-topic"],   # levels 0, 1, 2
    ),
    "complete": Score(
        instructions="Given `context`, how completely does `answer` cover what the question asks?",
        criteria=["Misses most of it", "Covers some of it", "Covers everything the context supports"],   # 0, 1, 2
    ),
    "refuses": Noul(
        instructions="Does `answer` decline, refuse, or say it cannot help, instead of attempting an answer?",
    ),
}
print("rubric:", list(RUBRIC))

rubric: ['faithful', 'relevant', 'complete', 'refuses']


## Step 2 · Grade one hallucinated answer, and read the raw response

The context says the tower is 330 m tall and was completed in 1889. The answer gets the height right, then invents a
designer (Thomas Edison) and a date (1901).

`jev.system_one(state=..., questions=RUBRIC)` sends the state plus **all four** questions in **one** request. The answers
come back grouped by type: `r.nouls[...]` and `r.scores[...]`. `r.usage.input_tokens` tells us what we were billed for.

In [3]:
CTX = ("The Eiffel Tower was completed in 1889 for the World's Fair. It is 330 metres tall "
       "including antennas and is located on the Champ de Mars in Paris.")

state = {"question": "How tall is the Eiffel Tower?", "context": CTX,
         "answer": "It is 330 metres tall and was designed by Thomas Edison in 1901."}
r = jev.system_one(state=state, questions=RUBRIC)

print(f"faithful: {r.nouls['faithful'].noul:.2f}")
print(f"relevant: {r.scores['relevant'].score:.2f} / 2   (confidence {r.scores['relevant'].confidence:.2f})")
print(f"complete: {r.scores['complete'].score:.2f} / 2   (confidence {r.scores['complete'].confidence:.2f})")
print(f"refuses : {r.nouls['refuses'].noul:.2f}")
print(f"tokens  : {r.usage.input_tokens}  ->  ${r.usage.input_tokens / 1e6 * PRICE_PER_M:.6f}")

faithful: 0.03
relevant: 1.36 / 2   (confidence 0.46)
complete: 1.64 / 2   (confidence 0.47)
refuses : 0.01
tokens  : 520  ->  $0.000022


**What to notice**
- `faithful` is near **0**: Jev caught the invented designer and date, even though the height is correct.
- `relevant` lands **between partial and full**, with a fairly **low confidence**: the answer does address the height, but
  half of it is about something else. Jev is telling you it is unsure where on the scale this belongs.
- `refuses` is near 0: the answer tried. It just made things up.
- One request, four answers, a few hundred tokens: a tiny fraction of a cent.

## Step 3 · The verdict, computed in code

Jev produced **numbers**. The verdict is ordinary Python you can read, test and change. Rules are checked **in order**,
and the **first** one that fires names the verdict:

| # | Rule | Verdict | Why this position |
|---|---|---|---|
| 1 | `refuses > 0.5` | `FAIL · non-answer` | if it didn't try, nothing else matters |
| 2 | `relevant < 0.5` (closer to *off-topic* than to *partial*) | `FAIL · off-topic` | it answered a different question |
| 3 | `faithful < 0.5` | `FAIL · hallucination` | it answered the question, but invented facts |
| 4 | `complete < 1.5` (closer to *some* than to *everything*) | `REVISE · incomplete` | true and on-topic, just short |
| 5 | otherwise | `PASS` | |

**Why order matters.** One bad answer often trips several rules at once. *"Paris is a wonderful city with great food and
museums"* is off-topic **and** unfaithful (food and museums are not in the context). Both are true, but the verdict
should name the failure you'd **fix first**: the bot didn't answer the question. So the rules go from "did it try?" to
"did it answer *this* question?" to "is what it said true?" to "is it enough?".

The order only picks the headline. We also return `flags`, **every** rule that fired, so nothing is hidden.

**The meta-eval.** If Jev's `confidence` on either `Score` is below `GRADE_CONFIDENT`, we set `review=True`: the grade
itself is uncertain, so a human should look.

> **Workshop note:** the first version checked faithfulness **first** and used `OFFTOPIC = 1.0`. Two problems showed up:
> the Paris answer was labelled `FAIL · hallucination` (true, but the wrong diagnosis), and `1.0` sits exactly on the
> *"partially addresses it"* level, so a partial answer scoring 0.99 counted as off-topic. Measured scores sit near
> **0** for off-topic answers and near **1** for partial ones, so the cut now sits halfway, at **0.5**, and the
> rules run in "what to fix first" order.

In [4]:
OFFTOPIC = 0.5          # relevant below this -> off-topic (halfway between level 0 and level 1)
COMP_OK = 1.5           # complete below this -> incomplete (halfway between level 1 and level 2)
GRADE_CONFIDENT = 0.55  # Score confidence below this -> a human should check the grade


def grade(question, context, answer, client):
    if "RUBRIC" not in globals():
        raise RuntimeError("RUBRIC is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(state={"question": question, "context": context, "answer": answer}, questions=RUBRIC)
    faithful, refuses = r.nouls["faithful"].noul, r.nouls["refuses"].noul
    relevant, complete = r.scores["relevant"], r.scores["complete"]

    # Every rule that fires, in priority order. The first one becomes the verdict.
    flags = []
    if refuses > 0.5:
        flags.append("FAIL · non-answer")
    if relevant.score < OFFTOPIC:
        flags.append("FAIL · off-topic")
    if faithful < 0.5:
        flags.append("FAIL · hallucination")
    if complete.score < COMP_OK:
        flags.append("REVISE · incomplete")
    verdict = flags[0] if flags else "PASS"

    grader_unsure = min(relevant.confidence, complete.confidence) < GRADE_CONFIDENT
    return {"verdict": verdict, "flags": flags, "faithful": faithful, "refuses": refuses,
            "relevant": relevant.score, "complete": complete.score,
            "review": grader_unsure, "tokens": r.usage.input_tokens}

## Step 4 · A small labelled eval set

Six answers to questions about the same context, each written to show one behaviour. `EXPECTED` is the verdict a human
grader would give. Having labels is what lets you **measure** the grader instead of trusting it.

| Answer | Expected | Why |
|---|---|---|
| "It is 330 metres tall including antennas." | PASS | correct, complete, grounded |
| "...designed by Thomas Edison in 1901." | hallucination | invented facts |
| "In 1889." (to *when **and why***) | incomplete | answers only half |
| "Paris is a wonderful city..." | off-topic | doesn't answer the question |
| "I'm not able to help with that." | non-answer | refusal |
| "It was completed in 1889 for the World's Fair." | PASS | both parts answered |

This cell only defines the data.

In [5]:
CASES = [
    ("How tall is the Eiffel Tower?", CTX, "It is 330 metres tall including antennas."),
    ("How tall is the Eiffel Tower?", CTX, "It is 330 metres tall and was designed by Thomas Edison in 1901."),
    ("When was the Eiffel Tower completed and why?", CTX, "In 1889."),
    ("How tall is the Eiffel Tower?", CTX, "Paris is a wonderful city with great food and museums."),
    ("Where is the Eiffel Tower?", CTX, "I'm not able to help with that."),
    ("When was the Eiffel Tower completed and why?", CTX, "It was completed in 1889 for the World's Fair."),
]
EXPECTED = ["PASS", "FAIL · hallucination", "REVISE · incomplete", "FAIL · off-topic", "FAIL · non-answer", "PASS"]
print(len(CASES), "cases")

6 cases


Now grade all six (one Jev call each), compare with the labels, and add up the cost. `✓`/`✗` shows agreement with the
label, `also:` lists any extra rules that fired, and `⚑ review` marks grades where Jev itself was unsure.

In [6]:
grades = [grade(q, c, a, jev) for q, c, a in CASES]

for (q, _, a), d, exp in zip(CASES, grades, EXPECTED):
    ok = "✓" if d["verdict"] == exp else "✗"
    flag = "  ⚑ review" if d["review"] else ""
    print(f"{ok} {d['verdict']:<22} {a[:52]:<52}{flag}")
    if len(d["flags"]) > 1:
        print(f"    also: {', '.join(d['flags'][1:])}")

agree = sum(d["verdict"] == e for d, e in zip(grades, EXPECTED))
tokens = sum(d["tokens"] for d in grades)
cost = tokens / 1e6 * PRICE_PER_M
print(f"\nagreement with labels: {agree}/{len(CASES)}")
print(f"cost to grade all {len(CASES)}: ${cost:.6f}  ->  about ${cost / len(CASES) * 1000:.3f} per 1,000 answers")

✓ PASS                   It is 330 metres tall including antennas.           
✓ FAIL · hallucination   It is 330 metres tall and was designed by Thomas Edi  ⚑ review
✓ REVISE · incomplete    In 1889.                                            
✓ FAIL · off-topic       Paris is a wonderful city with great food and museum
    also: FAIL · hallucination, REVISE · incomplete
✓ FAIL · non-answer      I'm not able to help with that.                     
    also: FAIL · off-topic, REVISE · incomplete
✓ PASS                   It was completed in 1889 for the World's Fair.      

agreement with labels: 6/6
cost to grade all 6: $0.000129  ->  about $0.021 per 1,000 answers


**What to notice**
- Every verdict matches its label, and each failure gets the **right diagnosis**: the Paris answer is **off-topic**, and
  its unfaithfulness still shows up under `also:`.
- The refusal also trips `off-topic` (a refusal doesn't address the question either). The order makes `non-answer` the
  headline, which is the more useful thing to tell the developer.
- The Edison answer is flagged **⚑ review**: Jev was unsure where a half-right, half-invented answer sits on the relevance
  and completeness scales. That is the meta-eval doing its job, pointing a human at the genuinely ambiguous grade.
- Grading costs around **two cents per thousand** answers, which is why you can grade all of them instead of sampling.

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a text box wired to `grade()` with `ipywidgets`, and
an HTML scorecard. The grading logic above is unchanged.

In [7]:
import ipywidgets as widgets
from IPython.display import HTML, display

# small style kit
INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:820px;">{inner}</div>'


def text_app(placeholder, render, button="Run", value=""):
    """A text box + button. On submit, call render(text) and show the HTML it returns."""
    inp = widgets.Text(value=value, placeholder=placeholder, continuous_update=False,
                       layout=widgets.Layout(width="68%"))
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            if inp.value.strip():
                display(HTML(box("<i>thinking...</i>")))
                html = render(inp.value.strip())
                out.clear_output(wait=True)
                display(HTML(html))

    btn.on_click(go)
    inp.observe(go, names="value")   # fires when you press Enter
    display(widgets.HBox([inp, btn]), out)

`scorecard_html()` turns a list of graded cases into cards: a coloured verdict badge, the three key numbers, extra flags,
and the review marker. It reuses the `grades` we already computed, so this cell makes no new Jev calls.

In [8]:
EV_COLOR = {"PASS": SAGE, "REVISE": AMBER, "FAIL": RED}


def scorecard_html(cases, grades):
    counts = {"PASS": 0, "REVISE": 0, "FAIL": 0}
    rows = ""
    for (q, _, a), d in zip(cases, grades):
        key = d["verdict"].split(" ")[0]
        counts[key] += 1
        col = EV_COLOR[key]
        flag = ' <span style="color:#C98A00;">&#9873; review</span>' if d["review"] else ""
        also = (f'<div style="color:{SLATE};font-size:12px;margin-top:4px;">also: {", ".join(d["flags"][1:])}</div>'
                if len(d["flags"]) > 1 else "")
        rows += (f'<div style="{CARD}border-left:5px solid {col};">'
                 f'<div style="font-size:13px;color:{SLATE};">Q: {q}</div><div style="margin:4px 0;">A: {a}</div>'
                 f'<span style="{BADGE}background:{col};">{d["verdict"]}</span>'
                 f'<span style="color:{SLATE};font-size:12px;"> &nbsp;faithful {d["faithful"]:.2f} · relevant {d["relevant"]:.2f}'
                 f' · complete {d["complete"]:.2f}</span>{flag}{also}</div>')
    summary = (f'<div style="{CARD}font-weight:600;">{counts["PASS"]} pass &nbsp; {counts["REVISE"]} revise &nbsp; '
               f'{counts["FAIL"]} fail &nbsp; (pass rate {counts["PASS"] / len(cases) * 100:.0f}%)</div>')
    return box(summary + rows)


display(HTML(scorecard_html(CASES, grades)))

## Step 6 · Try it yourself

Write your own answer to *"When was the Eiffel Tower completed and why?"*, graded against the same context. Press
**Enter** or click **Grade** (needs a live kernel). Good ones to try:
- `It was completed in 1889 for the World's Fair.` → **PASS**
- `It opened in 1889 and was built by aliens.` → **FAIL · hallucination** (on-topic, but invents a fact)
- `1889.` → **REVISE · incomplete** (answers *when*, not *why*)
- `I'd rather not say.` → **FAIL · non-answer**
- `The Louvre is the most visited museum in the world.` → **FAIL · off-topic**

In [9]:
QUESTION = "When was the Eiffel Tower completed and why?"
print("context :", CTX)
print("question:", QUESTION)
text_app("your answer, e.g. 'It opened in 1889 and was built by aliens.'",
         lambda ans: scorecard_html([(QUESTION, CTX, ans)], [grade(QUESTION, CTX, ans, jev)]), "Grade")

context : The Eiffel Tower was completed in 1889 for the World's Fair. It is 330 metres tall including antennas and is located on the Champ de Mars in Paris.
question: When was the Eiffel Tower completed and why?


Output()

## Recap and exercises

**Key ideas**
1. A rubric is just **typed questions**: `Noul` for yes/no checks, `Score` for graded scales. One Jev call answers them all.
2. **The model measures, the code decides.** Thresholds and rule order live in Python, where you can see and test them.
3. **Rule order is a design choice.** Put the failure you'd fix first at the top, and keep the other flags visible.
4. Jev's `confidence` gives you a **meta-eval** for free: send uncertain grades to a human, trust the rest.

**Exercises**
- Move the `faithful` rule back to the top of `grade()` and re-run Step 4. Which verdict changes, and is the new label more or less useful?
- Add a `concise` `Score` to `RUBRIC` (e.g. *"Rambling" / "OK" / "Tight"*) and a `REVISE · verbose` rule.
- Raise `GRADE_CONFIDENT` to 0.8. How many grades now go to review? What does that do to a human reviewer's workload?
- Write 10 answers of your own with labels and measure the agreement. Where Jev disagrees, is it the wording of a question or the threshold?